In [1]:
!python3 -m pip install groq

  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [groq]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from groq import Groq

In [3]:
load_dotenv()

True

In [4]:
client = chromadb.Client()

In [5]:
embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
            model_name="all-MiniLM-L6-v2"
        )

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 10169.57it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
collection = client.get_or_create_collection(
            name="qa_chat",
            embedding_function=embedder,
            metadata={"hnsw:space": "cosine"},
        )

In [8]:
data = pd.read_csv("data.csv")
data.head(4)

,question,answer
0,what Valaxy offering or who is valaxy?,Valaxy provides instructor-led training progra...
1,Who is the instructor giving training on AIML ...,PR Reddy is an experienced instructor speciali...
2,what are the top technologies covered in AIML ...,The course covers Python for Agentic AI and St...
3,What is the title of the course?,AI/ML Practitioner Training (AMPT).


In [10]:
questions = data["question"].astype(str).tolist()
questions[:5]

['what Valaxy offering or who is valaxy?',
 'Who is the instructor giving training on AIML course?',
 'what are the top technologies covered in AIML course?',
 'What is the title of the course?',
 'What is the primary goal of AMPT?']

In [11]:
answers = [{"answer": a} for a in data["answer"].astype(str).tolist()]
answers[:3]

[{'answer': 'Valaxy provides instructor-led training programs that empower the next generation of AI engineers.'},
 {'answer': 'PR Reddy is an experienced instructor specializing in the AI/ML technology stack.'},
 {'answer': 'The course covers Python for Agentic AI and Statistical Machine Learning and Deep Learning and Neural Networks and Generative AI and Agentic AI Frameworks and AWS Bedrock and LLMs and and MCP Protocol and among others.'}]

In [12]:
ids = [f"id_{i}" for i in range(len(questions))]
ids[:3]

['id_0', 'id_1', 'id_2']

In [13]:
collection.upsert(documents=questions, metadatas=answers, ids=ids)

In [14]:
collection.peek()

{'ids': ['id_0',
  'id_1',
  'id_2',
  'id_3',
  'id_4',
  'id_5',
  'id_6',
  'id_7',
  'id_8',
  'id_9'],
 'embeddings': array([[-0.03074309,  0.04349637, -0.0948014 , ..., -0.07201263,
          0.03838382,  0.06142864],
        [-0.0314416 ,  0.02390029, -0.060661  , ..., -0.03050497,
         -0.05160307,  0.02039726],
        [-0.03319364, -0.00414791, -0.00904497, ..., -0.06237044,
         -0.04239853,  0.12312423],
        ...,
        [ 0.06443509,  0.01176433,  0.045216  , ..., -0.04429111,
          0.03667981,  0.01157078],
        [-0.05849819,  0.00979794,  0.05321462, ...,  0.00689104,
         -0.0113435 , -0.05680015],
        [-0.05492701, -0.02286353, -0.00649674, ...,  0.0180149 ,
          0.16810329,  0.00656873]], shape=(10, 384)),
 'documents': ['what Valaxy offering or who is valaxy?',
  'Who is the instructor giving training on AIML course?',
  'what are the top technologies covered in AIML course?',
  'What is the title of the course?',
  'What is the primar

In [15]:
query = "how much is the AIML course fee?"

In [16]:
def _normalize_query(q: str) -> str:
        return q.strip().rstrip(" ?!.")
query = _normalize_query(query)

In [17]:
query

'how much is the AIML course fee'

In [18]:
q_emb = embedder([query])[0]

In [19]:
results = collection.query(query_embeddings=[q_emb], n_results=5)

In [20]:
results

{'ids': [['id_90', 'id_94', 'id_1', 'id_2', 'id_70']],
 'embeddings': None,
 'documents': [['What will be the AIML course fee?',
   'AIML course duration please?',
   'Who is the instructor giving training on AIML course?',
   'what are the top technologies covered in AIML course?',
   'How do I enroll in a course?']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'answer': 'The regular fee is INR 30 and000. Contact the admin team for available discount coupons.'},
   {'answer': 'The total course duration is 90 days.'},
   {'answer': 'PR Reddy is an experienced instructor specializing in the AI/ML technology stack.'},
   {'answer': 'The course covers Python for Agentic AI and Statistical Machine Learning and Deep Learning and Neural Networks and Generative AI and Agentic AI Frameworks and AWS Bedrock and LLMs and and MCP Protocol and among others.'},
   {'answer': 'Create an account and browse available courses and and click ‘Enrol

In [22]:
documents = results['documents'][0]

In [23]:
documents

['What will be the AIML course fee?',
 'AIML course duration please?',
 'Who is the instructor giving training on AIML course?',
 'what are the top technologies covered in AIML course?',
 'How do I enroll in a course?']

In [24]:
metadatas = results['metadatas'][0]

In [25]:
metadatas

[{'answer': 'The regular fee is INR 30 and000. Contact the admin team for available discount coupons.'},
 {'answer': 'The total course duration is 90 days.'},
 {'answer': 'PR Reddy is an experienced instructor specializing in the AI/ML technology stack.'},
 {'answer': 'The course covers Python for Agentic AI and Statistical Machine Learning and Deep Learning and Neural Networks and Generative AI and Agentic AI Frameworks and AWS Bedrock and LLMs and and MCP Protocol and among others.'},
 {'answer': 'Create an account and browse available courses and and click ‘Enroll’ or ‘Start Learning’.'}]

In [26]:
pairs = []
for d, m in zip(documents, metadatas):
    a = (m.get("answer", "") if isinstance(m, dict) else "")
    if d or a:
        pairs.append(f"Q: {d}\nA: {a}")

context = "\n\n".join(pairs).strip()
context

'Q: What will be the AIML course fee?\nA: The regular fee is INR 30 and000. Contact the admin team for available discount coupons.\n\nQ: AIML course duration please?\nA: The total course duration is 90 days.\n\nQ: Who is the instructor giving training on AIML course?\nA: PR Reddy is an experienced instructor specializing in the AI/ML technology stack.\n\nQ: what are the top technologies covered in AIML course?\nA: The course covers Python for Agentic AI and Statistical Machine Learning and Deep Learning and Neural Networks and Generative AI and Agentic AI Frameworks and AWS Bedrock and LLMs and and MCP Protocol and among others.\n\nQ: How do I enroll in a course?\nA: Create an account and browse available courses and and click ‘Enroll’ or ‘Start Learning’.'

In [27]:
prompt = f"""
    You are a helpful FAQ assistant. Use ONLY the context to answer.
    Rewrite the answer in clear, friendly language (not verbatim), and format it nicely.
    - If the question asks for "course content", "course index", or similar: present a clean, ordered outline.
    - If context includes lists, use bullets or numbered steps.
    - If dates, prices, or times appear, surface them clearly (you may bold them).
    - If multiple Q/A pairs are relevant, synthesize them into one concise answer.
    - At the end of the answer, mention the following line only if the user’s query is about pricing: Are you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!
    If the answer is not in the context, reply exactly: "I don't know".

    Question: {query}

    Context:
    {context}
    """

In [28]:
prompt

'\n    You are a helpful FAQ assistant. Use ONLY the context to answer.\n    Rewrite the answer in clear, friendly language (not verbatim), and format it nicely.\n    - If the question asks for "course content", "course index", or similar: present a clean, ordered outline.\n    - If context includes lists, use bullets or numbered steps.\n    - If dates, prices, or times appear, surface them clearly (you may bold them).\n    - If multiple Q/A pairs are relevant, synthesize them into one concise answer.\n    - At the end of the answer, mention the following line only if the user’s query is about pricing: Are you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!\n    If the answer is not in the context, reply exactly: "I don\'t know".\n\n    Question: how much is the AIML course fee\n\n    Context:\n    Q: What will be the AIML course fee?\nA: The regular fee is INR 30 and000. Contact the admin team for availabl

In [29]:
groq = Groq()
llm_answer = groq.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
        )

In [30]:
llm_answer

ChatCompletion(id='chatcmpl-d460c280-7198-4a1b-9ed5-760fab233bf0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**AIML Course Fee:**\nThe regular fee for the AIML course is **INR 30,000**. However, you can contact the admin team to check if there are any available discount coupons.\n\nAre you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))], created=1775702725, model='llama-3.3-70b-versatile', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_d42c28f9ce', usage=CompletionUsage(completion_tokens=72, prompt_tokens=406, total_tokens=478, completion_time=0.15496612, completion_tokens_details=None, prompt_time=0.03344277, prompt_tokens_details=None, queue_time=0.04755178, total_time=0.18840889), usage

In [31]:
llm_answer.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**AIML Course Fee:**\nThe regular fee for the AIML course is **INR 30,000**. However, you can contact the admin team to check if there are any available discount coupons.\n\nAre you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None))

In [32]:
answer = llm_answer.choices[0].message.content

In [33]:
answer

'**AIML Course Fee:**\nThe regular fee for the AIML course is **INR 30,000**. However, you can contact the admin team to check if there are any available discount coupons.\n\nAre you interested in joining the AIML course to become an AI Expert? I can check if there are any special discounts available for you!'